# 04 · Pricing GLMs: Frequency, Severity and Total Loss

Notebook 03 compared BEV and ICE one factor at a time. Here we fit three GLMs that control for all rating factors at once, and turn the BEV coefficients into tariff relativities.

| Model | Response | Family / link | Relativity |
|---|---|---|---|
| 1. Frequency | own-damage claim count per policy | Poisson / log, offset $\log(\text{exposure})$ | rate ratio $e^{\beta}$ |
| 2. Severity | own-damage cost per claim | Gamma / log | cost ratio $e^{\beta}$ |
| 3. Total loss | 1 if the claim is a total loss | Binomial / logit | odds ratio $e^{\beta}$ |

**Data rules** (same as notebooks 02–03): claims are joined many-to-one on `Policy_ID`, must fall inside the policy period, and the 2025-01-19 storm is excluded. An own-damage claim has `Own_Damage_Amount > 0`. **Scope: BEV vs. ICE only.** Hybrids are removed at the start, and `Powertrain` has two levels with ICE as the reference. Rows with a missing rating variable are dropped by `glm()`.

## 1. Setup and Data

In [1]:
suppressWarnings(suppressPackageStartupMessages({
  library(tidyverse)
  library(scales)
  library(broom)
}))

fmt_p <- function(p) if_else(p < 0.001, "< 0.001", number(p, accuracy = 0.001))
p_str <- function(p) if_else(p < 0.001, "p < 0.001", paste("p =", number(p, accuracy = 0.001)))

# Path to a file in data/processed/; works from notebooks/ or the repository root
processed_path <- function(file) {
  path <- file.path(c("../data/processed", "data/processed"), file) |> keep(file.exists) |> first()
  if (is.na(path)) stop("Cannot find ", file, " (run src/01_clean_datasets.R first)")
  path
}

# BEV vs. ICE only; ICE is the reference level
policies <- read_rds(processed_path("policies_cleaned.rds")) |>
  filter(Powertrain %in% c("ICE", "BEV")) |>
  mutate(Powertrain = relevel(factor(as.character(Powertrain)), ref = "ICE"))

# Own-damage claims inside the policy period, storm excluded (join checked in src/02_verify_join.R)
od_claims <- read_rds(processed_path("claims_cleaned.rds")) |>
  filter(Own_Damage_Amount > 0, !is_cat_event) |>
  inner_join(select(policies, Policy_ID, Start_Date, End_Date, Powertrain, Vehicle_Value, Vehicle_Age,
                    Driver_Age, District),
             by = "Policy_ID", relationship = "many-to-one") |>
  filter(between(Accident_Date, Start_Date, End_Date))

# Model 1: claim count per policy (0 if none), exposure in vehicle-years
freq_data <- policies |>
  left_join(count(od_claims, Policy_ID, name = "Claim_Count"), by = "Policy_ID") |>
  mutate(Claim_Count = coalesce(Claim_Count, 0L))

# Model 2: drop the amounts above 2x vehicle value (entry errors)
sev_data <- od_claims |> filter(Own_Damage_Amount <= 2 * Vehicle_Value)

# Model 3: theft excluded (a stolen car is written off whatever its powertrain)
tl_data <- od_claims |> filter(Claim_Type != "Theft")

tibble(model = c("1. Frequency (policies)", "2. Severity (claims)", "3. Total loss (claims)"),
       rows  = c(nrow(freq_data), nrow(sev_data), nrow(tl_data)))

model,rows
<chr>,<int>
1. Frequency (policies),30046
2. Severity (claims),4281
3. Total loss (claims),4203


In [2]:
# Relativity table: exp(beta) with a 95% Wald CI. For log(Vehicle_Value) we show 2^beta,
# i.e. the factor when the vehicle value doubles.
relativities <- function(fit) {
  tidy(fit) |>
    filter(term != "(Intercept)") |>
    mutate(base = if_else(term == "log(Vehicle_Value)", 2, exp(1)),
           relativity = base^estimate,
           ci_low  = base^(estimate - 1.96 * std.error),
           ci_high = base^(estimate + 1.96 * std.error)) |>
    transmute(term = if_else(term == "log(Vehicle_Value)", "Vehicle_Value (per doubling)", term),
              beta = number(estimate, 0.001), relativity = number(relativity, 0.001),
              ci_95 = paste0(number(ci_low, 0.001), " – ", number(ci_high, 0.001)),
              p_value = fmt_p(p.value))
}

# BEV coefficient of a fitted model: estimate, standard error, p-value
bev <- function(fit) tidy(fit) |> filter(term == "PowertrainBEV")

## 2. Model 1: Claim Frequency (Poisson GLM)
$$\log E[N_i] = \log(\text{exposure}_i) + \beta_0 + \beta_{\text{BEV}} +\beta_v \log(\text{Value}_i) + \beta_a \,\text{VehAge}_i + \beta_d \,\text{DriverAge}_i$$

In [3]:
freq_fit <- glm(Claim_Count ~ Powertrain + log(Vehicle_Value) + Vehicle_Age + Driver_Age,
                offset = log(exposure), family = poisson(link = "log"), data = freq_data)

relativities(freq_fit)

dispersion <- sum(residuals(freq_fit, type = "pearson")^2) / df.residual(freq_fit)
b1 <- bev(freq_fit)
b1_qp <- bev(update(freq_fit, family = quasipoisson(link = "log")))   # same fit, SEs scaled by the dispersion
cat("Policies used:", nobs(freq_fit), "| Pearson dispersion:", number(dispersion, 0.001),
    "| BEV p-value with quasi-Poisson SEs:", fmt_p(b1_qp$p.value), "\n")
cat("Takeaway: BEV frequency relativity =", number(exp(b1$estimate), 0.001), paste0("(", p_str(b1$p.value), ")"),
    if_else(b1$p.value < 0.05, "-> differs significantly from 1.00", "-> not significantly different from 1.00"), "\n")

term,beta,relativity,ci_95,p_value
<chr>,<chr>,<chr>,<chr>,<chr>
PowertrainBEV,0.108,1.114,1.013 – 1.224,0.026
Vehicle_Value (per doubling),-0.178,0.884,0.834 – 0.937,< 0.001
Vehicle_Age,-0.015,0.985,0.974 – 0.996,0.009
Driver_Age,-0.015,0.985,0.983 – 0.987,< 0.001


Policies used: 29212 | Pearson dispersion: 1.099 | BEV p-value with quasi-Poisson SEs: 0.034 


Takeaway: BEV frequency relativity = 1.114 (p = 0.026) -> differs significantly from 1.00 


**Result:**
- **BEV:** relativity **1.11** (95% CI 1.01–1.22, $p = 0.026$).
- **Overdispersion:** the Pearson dispersion is 1.10. With quasi-Poisson standard errors the BEV p-value is 0.034, so the result holds.
- **Why this is higher than notebook 03 (RR 1.05, not significant):** more expensive cars have *fewer* claims (×0.88 per doubling of value). Once value is held fixed, the BEV frequency comes out about 11% higher than ICE.

**Takeaway:** at equal value, vehicle age and driver age, BEVs claim about 11% more often than ICE cars.

## 3. Model 2: Own-Damage Severity (Gamma GLM)
$$\log E[X_j] = \beta_0 + \beta_{\text{BEV}} +\beta_v \log(\text{Value}_j) + \beta_a \,\text{VehAge}_j + \gamma_{\text{District}}$$
The Gamma family fits right-skewed, positive costs with a constant coefficient of variation; with the log link every coefficient is a multiplicative cost factor.

In [4]:
sev_fit <- glm(Own_Damage_Amount ~ Powertrain + log(Vehicle_Value) + Vehicle_Age + District,
               family = Gamma(link = "log"), data = sev_data)

relativities(sev_fit)

b2 <- bev(sev_fit)
cat("Claims used:", nobs(sev_fit), "\n")
cat("Takeaway: BEV severity relativity =", number(exp(b2$estimate), 0.001),
    paste0("(95% CI ", number(exp(b2$estimate - 1.96 * b2$std.error), 0.001), " – ",
           number(exp(b2$estimate + 1.96 * b2$std.error), 0.001), ", ", p_str(b2$p.value), ")"),
    if_else(b2$p.value < 0.05, "-> significant", "-> not significant"), "\n")

term,beta,relativity,ci_95,p_value
<chr>,<chr>,<chr>,<chr>,<chr>
PowertrainBEV,0.150,1.162,1.022 – 1.320,0.022
Vehicle_Value (per doubling),0.724,1.652,1.531 – 1.783,< 0.001
Vehicle_Age,0.013,1.013,0.998 – 1.029,0.084
DistrictHaifa,-0.071,0.931,0.807 – 1.074,0.328
DistrictJerusalem,-0.104,0.902,0.786 – 1.035,0.141
DistrictJudea & Samaria,-0.298,0.742,0.508 – 1.084,0.123
DistrictNorth,0.009,1.009,0.874 – 1.165,0.904
DistrictSouth,0.074,1.077,0.943 – 1.229,0.273
DistrictTel Aviv,-0.046,0.955,0.848 – 1.076,0.452


Claims used: 4281 


Takeaway: BEV severity relativity = 1.162 (95% CI 1.022 – 1.320, p = 0.022) -> significant 


**Result:**
- **BEV:** relativity **1.16** (95% CI 1.02–1.32, $p = 0.022$).
- **Vehicle value:** the strongest cost driver. A car worth twice as much has claims about 1.65× as expensive, an elasticity of 0.72.
- **District:** no district differs significantly from Center.
- **Why this is below notebook 03's 1.29:** that figure was a ratio of *geometric* means inside the age ≤ 5 cohort with three value tiers. The Gamma GLM models the *mean* cost on all ages with continuous value.

**Takeaway:** at equal value, age and district, a BEV own-damage claim costs about 16% more than an ICE claim.

## 4. Model 3: Total-Loss Probability (Logistic GLM)
$$\text{logit}\, P(\text{TL}_j = 1) = \beta_0 + \beta_{\text{BEV}} +\beta_v \log(\text{Value}_j) + \beta_a \,\text{VehAge}_j$$

In [5]:
tl_fit <- glm(is_total_loss_clean ~ Powertrain + log(Vehicle_Value) + Vehicle_Age,
              family = binomial(link = "logit"), data = tl_data)

relativities(tl_fit)

b3 <- bev(tl_fit)
cat("Claims used:", nobs(tl_fit), "| total losses:", sum(tl_fit$y), "\n")
cat("Takeaway: BEV total-loss odds ratio =", number(exp(b3$estimate), 0.001),
    paste0("(95% CI ", number(exp(b3$estimate - 1.96 * b3$std.error), 0.001), " – ",
           number(exp(b3$estimate + 1.96 * b3$std.error), 0.001), ", ", p_str(b3$p.value), ")"),
    if_else(b3$p.value < 0.05, "-> significant", "-> not significant"), "\n")

term,beta,relativity,ci_95,p_value
<chr>,<chr>,<chr>,<chr>,<chr>
PowertrainBEV,0.688,1.990,1.409 – 2.811,< 0.001
Vehicle_Value (per doubling),-0.523,0.696,0.566 – 0.855,< 0.001
Vehicle_Age,0.021,1.022,0.985 – 1.060,0.254


Claims used: 4150 | total losses: 340 


Takeaway: BEV total-loss odds ratio = 1.990 (95% CI 1.409 – 2.811, p < 0.001) -> significant 


**Result:**
- **BEV:** odds ratio **1.99** (95% CI 1.41–2.81, $p < 0.001$).
- **Vehicle value:** higher-value cars are written off less often (×0.70 per doubling of value), because repairs are a smaller share of a more valuable car.
- **Why this is clearer than the crude tests:** BEVs are newer and pricier, which hides their higher write-off rate in a raw comparison (notebook 03).

**Takeaway:** at equal value and age, a damaged BEV has about twice the odds of being written off, which supports the battery-damage argument.

## 5. Tariff Formula: Pure Premium Relativity
The general commercial tariff formula from the seminar syllabus:
$$\text{Commercial Premium} = (\mathbb{E}[\text{Frequency}] \times \mathbb{E}[\text{Severity}]) \times \text{Risk Factors} + \text{Expenses} + \text{Risk Margin and Profit}$$
- **$\mathbb{E}[\text{Frequency}]$ (expected frequency):** modelled with the Poisson GLM and its exposure offset (Model 1). It is the annual claim frequency per policy.
- **$\mathbb{E}[\text{Severity}]$ (expected severity):** modelled with the Gamma GLM with log link (Model 2). It is the average cost of an attritional own-damage claim.
- **$\mathbb{E}[\text{Frequency}] \times \mathbb{E}[\text{Severity}]$ (pure premium):** the baseline risk cost needed to cover expected claims.
- **Risk factors (relativities):** multiplicative adjustments for specific risks. Here that is our BEV factor, calculated below ($\text{Relativity}_{\text{BEV}} \approx 1.29$).
- **Expenses, risk margin and profit:** commercial loadings set by the insurer (distribution, administration, cost of capital). They are outside what claims data can estimate.

Both GLMs use a log link, so the powertrain relativities multiply:
$$\text{Pure Premium Relativity}_{\text{BEV}} = \text{Relativity}_{\text{Freq}} \times \text{Relativity}_{\text{Sev}} = e^{\beta^{\text{freq}}_{\text{BEV}} + \beta^{\text{sev}}_{\text{BEV}}}$$
Treating the two models as independent, the 95% CI uses $\text{se} = \sqrt{\text{se}_{\text{freq}}^2 + \text{se}_{\text{sev}}^2}$ on the log scale.

Total losses are already part of the own-damage cost in Model 2, so the total-loss model is **not** multiplied in again. That would count the same losses twice.

In [6]:
f  <- bev(freq_fit)
s  <- bev(sev_fit)
se <- sqrt(f$std.error^2 + s$std.error^2)

pure_premium <- tibble(
  Powertrain   = c("ICE (base)", "BEV"),
  frequency    = c(1, exp(f$estimate)),
  severity     = c(1, exp(s$estimate)),
  pure_premium = c(1, exp(f$estimate + s$estimate)),
  ci_low       = c(NA, exp(f$estimate + s$estimate - 1.96 * se)),
  ci_high      = c(NA, exp(f$estimate + s$estimate + 1.96 * se)))

pure_premium |>
  mutate(across(frequency:pure_premium, ~ number(.x, 0.001)),
         ci_95 = if_else(is.na(ci_low), "-", paste0(number(ci_low, 0.001), " – ", number(ci_high, 0.001))),
         .keep = "unused")

Powertrain,frequency,severity,pure_premium,ci_95
<chr>,<chr>,<chr>,<chr>,<chr>
ICE (base),1.000,1.000,1.000,-
BEV,1.114,1.162,1.293,1.103 – 1.517


**Result:**
$$\text{PP}_{\text{BEV}} = 1.114 \times 1.162 = \mathbf{1.29} \quad (95\%\text{ CI } 1.10\text{–}1.52)$$
- The own-damage pure premium for a BEV is about 29% above an ICE car of the same value and age; ICE is the base at 1.00.
- About 11 percentage points come from frequency and about 16 from severity.

**Takeaway:** the tariff should carry a BEV relativity of about 1.29 on the own-damage premium, on top of the vehicle-value factor.

## 6. Pricing and Underwriting Recommendations

| Component | Model | BEV relativity (ICE = 1.00) | Tariff impact | Underwriting recommendation |
|---|---|---|---|---|
| Frequency | Poisson GLM | 1.11 (1.01–1.22), $p = 0.026$ | Part of the combined BEV factor | Monitor as BEV exposure grows; re-estimate each year |
| Own-damage severity | Gamma GLM | 1.16 (1.02–1.32), $p = 0.022$ | Part of the combined BEV factor | Repair-network agreements for BEVs; review deductibles |
| **Own-damage pure premium** | Frequency × severity | **1.29** (1.10–1.52) | **BEV own-damage relativity ≈ 1.29**, on top of vehicle value | Main pricing change |
| Total loss | Logistic GLM | OR 1.99 (1.41–2.81), $p < 0.001$ | **No separate loading**: total losses are already in the severity model, so a second loading would double count | Battery inspection after impacts; specialist battery-repair and salvage partners to avoid write-offs |
| Third-party severity | t-test (notebook 03) | 0.95, $p = 0.42$ | 1.00: no change | Keep the third-party rate for BEVs the same as for ICE |

**Final tariff (own-damage part, multiplicative):**
$$\text{Premium} = \text{Base} \times f(\text{Vehicle Value}) \times f(\text{Vehicle Age}) \times f(\text{Driver Age}) \times \underbrace{\{1.00 \text{ ICE},\ 1.29 \text{ BEV}\}}_{\text{powertrain}} + \text{Expenses and Risk Margin}$$

Our GLMs estimate only the risk part of this formula, the pure premium. Expenses and profit margin are company-specific loadings set outside the model and added on top to reach the commercial premium.

**Limitations:**
- About 4,000 BEV vehicle-years and about 600 BEV claims, so each factor has a wide confidence interval.
- No large-loss cap is applied yet: the audit suggests capping own damage at P99.5 as a sensitivity check.
- The storm is excluded, so a separate CAT loading is needed. The storm barely changes the BEV/ICE frequency ratio (1.057 with it, 1.059 without; data quality audit §6).